In [2]:
import numpy as np
import os.path as op

bids_folder = '/Volumes/mrenkeED/data/ds-stressrisk'

In [ ]:
from utils import get_basic_mask, cleanTS

mask, labeling_noParcel = get_basic_mask()
clean_ts = cleanTS('01', 1,bids_folder=bids_folder)

In [4]:
from nilearn.connectome import ConnectivityMeasure

seed_ts = clean_ts[mask]
correlation_measure = ConnectivityMeasure(kind='correlation')
print('raw connectivity matrix estimated')

raw connectivity matrix estimated


In [5]:
from scipy.sparse.csgraph import connected_components

    # filter out nodes that are not connected to the rest
graph = correlation_measure.fit_transform([seed_ts.T])[0] #correlation_matrix
cc = connected_components(graph)
mask_cc = cc[1] == 0 # all nodes in 0 belong to the largest connected component, check #-components in cc[0]
mask[mask == True] = mask_cc # mark nodes not in component 0  as False in mask
print('mask with connected components created')


mask with connected components created


In [6]:
from brainspace.gradient import GradientMaps

seed_ts = clean_ts[mask]

#now perform embedding on cleaned data
correlation_measure = ConnectivityMeasure(kind='correlation')
correlation_matrix = correlation_measure.fit_transform([seed_ts.T])[0]
gm = GradientMaps(n_components=2, random_state=0)
gm.fit(correlation_matrix)


/Users/mrenke/mambaforge/envs/numrefields/lib/python3.10/site-packages/brainspace-0.1.4-py3.10.egg/brainspace/gradient/embedding.py:70: UserWarning: Affinity is not symmetric. Making symmetric.
  warnings.warn('Affinity is not symmetric. Making symmetric.')


GradientMaps(n_components=2, random_state=0)

In [11]:
print(mask.shape)
print(mask[mask==True].shape)


(20484,)
(18709,)


In [8]:
seed_ts.shape

(18709, 810)

In [9]:
gm.gradients_.shape

(18709, 2)

In [13]:
correlation_matrix.shape

(18709, 18709)

In [15]:
s = np.array([correlation_matrix, mask])

/var/folders/3k/8g0xv78x051fznwyh_m5xcn8f91w3q/T/ipykernel_52093/2894340987.py:1: VisibleDeprecationWarning: Creating an ndarray from ragged nested sequences (which is a list-or-tuple of lists-or-tuples-or ndarrays with different lengths or shapes) is deprecated. If you meant to do this, you must specify 'dtype=object' when creating the ndarray.
  s = np.array([correlation_matrix, mask])


In [20]:
s[1][s[1]==True].shape

(18709,)

In [22]:
sub='01'
ses=1
space='fsaverage5'
task = 'magjduge'

ex_file = op.join(bids_folder,'derivatives', 'fmriprep', f'sub-{sub}', f'ses-{ses}', 'func', f'sub-{sub}_ses-{ses}_task-{task}_run-1_space-{space}_hemi-L_bold.func.gii')


In [23]:
import os
os.path.exists(ex_file)

False